# Linear Regression Project Template
## Simple & Multiple Linear Regression in Python

**Reusable template** based on standard applied statistics workflows (least-squares estimation, R² / Adjusted R², model specification, context-dependency of coefficients).

**How to use:**  
1. Replace placeholder data paths / column names.  
2. Run cells sequentially.  
3. Document assumptions, diagnostics, and interpretation carefully.  
4. Always interpret coefficients *in the context of the full model*.

---


## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Optional: for reading SPSS files
# import pyreadstat

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline


## 2. Load & Inspect Data

Replace the path and column names with your own dataset.


In [ ]:
# Example: df = pd.read_csv('your_data.csv')
# Or: df, meta = pyreadstat.read_sav('iq_data.sav')

# Placeholder synthetic data for template demonstration
np.random.seed(42)
n = 100
df = pd.DataFrame({
    'quant': np.random.normal(50, 10, n),
    'analytic': np.random.normal(55, 12, n),
    'verbal': 30 + 0.5 * np.random.normal(50, 10, n) + 0.2 * np.random.normal(55, 12, n) + np.random.normal(0, 8, n),
    'group': np.random.choice([0, 1], n)
})

print(df.head())
print('
Shape:', df.shape)
print('
Missing values:
', df.isnull().sum())
print('
Descriptive statistics:
', df.describe().round(2))


## 3. Exploratory Data Analysis

- Visualize relationships
- Check linearity, outliers, distributions


In [ ]:
# Pairplot / correlation matrix
sns.pairplot(df[['verbal', 'quant', 'analytic']], diag_kind='kde')
plt.suptitle('Pairwise Relationships', y=1.02)
plt.show()

corr = df[['verbal', 'quant', 'analytic']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix')
plt.show()


In [ ]:
# Scatter + simple trend
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.regplot(x='quant', y='verbal', data=df, ax=axes[0], scatter_kws={'alpha':0.6})
axes[0].set_title('Verbal ~ Quant')
sns.regplot(x='analytic', y='verbal', data=df, ax=axes[1], scatter_kws={'alpha':0.6})
axes[1].set_title('Verbal ~ Analytic')
plt.tight_layout()
plt.show()


## 4. Simple Linear Regression

Fit:  using **ordinary least squares**.

Remember: the least-squares line minimizes ∑(yᵢ − ŷᵢ)².


In [ ]:
# Prepare design matrix (add constant for intercept)
X_simple = sm.add_constant(df['quant'])
y = df['verbal']

model_simple = sm.OLS(y, X_simple).fit()
print(model_simple.summary())


### Interpretation notes (Simple Regression)
- **Intercept**: predicted verbal when quant = 0 (often not substantively meaningful).
- **Slope**: expected change in verbal for a 1-unit increase in quant.
- **R²**: proportion of variance in verbal accounted for by quant.
- **Adj. R²**: penalized for model complexity (more useful when comparing models).
- Always report: “given the model under test”.


## 5. Multiple Linear Regression

Fit: 

**Critical concept**: coefficients are *partial* regression coefficients.  
They must be interpreted **holding the other predictors constant** / **in the context of the full model**.


In [ ]:
X_multi = df[['quant', 'analytic']]
X_multi = sm.add_constant(X_multi)

model_multi = sm.OLS(y, X_multi).fit()
print(model_multi.summary())


In [ ]:
# Using scikit-learn (for prediction-focused workflows)
regr = LinearRegression()
regr.fit(df[['quant', 'analytic']], y)
print('Intercept:', regr.intercept_)
print('Coefficients (quant, analytic):', regr.coef_)


### Interpretation notes (Multiple Regression)
- Coefficient for quant: expected change in verbal for +1 quant **while holding analytic constant**.
- Context-dependency: removing or adding a predictor can change the other coefficients.
- Model specification error (omitted variables) biases estimates.
- “All models are wrong, some are useful” – George Box.


## 6. Model Diagnostics & Assumptions

- Residuals vs fitted (linearity / heteroscedasticity)
- Q-Q plot (normality of residuals)
- VIF (multicollinearity)
- Influence / outliers


In [ ]:
# Residuals vs Fitted
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.residplot(x=model_multi.fittedvalues, y=model_multi.resid, lowess=True, ax=axes[0],
              scatter_kws={'alpha':0.5})
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')
axes[0].axhline(0, color='red', linestyle='--')

# Q-Q plot
sm.qqplot(model_multi.resid, line='45', ax=axes[1])
axes[1].set_title('Normal Q-Q')
plt.tight_layout()
plt.show()


In [ ]:
# Variance Inflation Factors (multicollinearity)
X_vif = df[['quant', 'analytic']].copy()
X_vif = sm.add_constant(X_vif)
vif_data = pd.DataFrame()
vif_data['Variable'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif_data)
# Rule of thumb: VIF > 5 or 10 indicates potential multicollinearity concern


## 7. Predictions & New Data

Always specify the values of *all* predictors in the model.


In [ ]:
# Example prediction
new_data = pd.DataFrame({'const': [1.0], 'quant': [20], 'analytic': [25]})
pred = model_multi.predict(new_data)
print('Predicted verbal:', pred.values[0].round(2))

# Or with sklearn
print('sklearn prediction:', regr.predict([[20, 25]])[0].round(2))


## 8. Model Comparison & Selection Notes

- Prefer simultaneous (full-entry) regression when theory guides predictors.
- Forward / Backward / Stepwise are algorithmic; use with caution (biased SEs, “fishing”).
- Always prioritize **candidate predictor selection** based on substantive knowledge over pure algorithms.
- Report both R² and Adjusted R².
- Consider AIC / BIC for nested or non-nested comparison (smaller is better).


In [ ]:
print('Simple model Adj. R²:', model_simple.rsquared_adj.round(3))
print('Multiple model Adj. R²:', model_multi.rsquared_adj.round(3))
print('Simple AIC:', model_simple.aic.round(1))
print('Multiple AIC:', model_multi.aic.round(1))


## 9. Reporting Checklist (fill in)

- [ ] Research question clearly stated
- [ ] Candidate predictors justified by theory / literature
- [ ] Model equation written (population & sample)
- [ ] Assumptions checked and reported
- [ ] Coefficients interpreted *in context of the model*
- [ ] R² and Adj. R² reported with clear meaning
- [ ] Limitations (specification error, omitted variables, generalizability) discussed
- [ ] No causal claims from purely observational data without strong design


---
**End of template**

Remember:  
> “In statistics, the word ‘predict’ only means as much as how it is defined mathematically.”

Always pair statistical results with strong research design and transparent model specification.
